# Irish Building Energy Ratings - Complete Analysis

**Dataset:** Every BER cert ever issued in Ireland - about 1.4 million dwellings with 211 fields each, scraped from SEAI's National BER Research Tool and flattened into a clean CSV and Parquet.

**Author:** Fionn Hughes

---

## What this notebook covers

1. **Annual Heating Cost Per BER Grade** - what each rating actually costs you per year
2. **The Rating Distribution** - what share of Irish homes sit at each letter grade
3. **The 2008 Building Regs Cliff** - the year regs got dramatically stricter
4. **Apartments vs Detached Houses** - shared walls save a lot of energy
5. **CO₂ Emissions Across the Decades** - how much carbon a typical home emits per m², by build era
6. **County Leaderboard** - most and least efficient counties
7. **Fuel Type by Build Decade** - the slow shift away from oil and solid fuel
8. **Wall Insulation Across a Century** - average wall U-values by decade built
9. **The Worst-Rated Homes** - where Ireland's F+G stock lives

Heads up: the BER scale changes on 24 May 2026 (going from 15 grades to flat A-G plus a new A0). This dataset is the old version captured right before SEAI swaps it out.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np

# running on kaggle.com - use this
#DATA_PATH = '/kaggle/input/datasets/fionnhughes/irish-building-energy-ratings/building_energy_ratings.parquet'

# running on your laptop - use this
DATA_PATH = '../datasets/03_building_energy_ratings/output/building_energy_ratings/building_energy_ratings.parquet'

df = pd.read_parquet(DATA_PATH)

print(df.head())

In [ ]:
# strip whitespaces and parse a couple of columns properly
df['countyname'] = df['countyname'].str.strip()
df['energyrating'] = df['energyrating'].str.strip()
df['mainspaceheatingfuel'] = df['mainspaceheatingfuel'].str.strip()
df['dateofassessment'] = pd.to_datetime(df['dateofassessment'], errors='coerce')
df['year_of_construction'] = pd.to_numeric(df['year_of_construction'], errors='coerce')
df['berrating'] = pd.to_numeric(df['berrating'], errors='coerce')
df['uvaluewall'] = pd.to_numeric(df['uvaluewall'], errors='coerce')

# strip 'Co. ' prefix from county names so the chart labels are cleaner
df['county_clean'] = df['countyname'].str.replace('Co. ', '', regex=False).str.strip()

# bucket fuels into a handful of useful categories - the raw column has a long
# tail of weird values like 'Bulk LPG (propane or butane)' that we collapse here
def bucket_fuel(f):
    if pd.isna(f):
        return 'Unknown'
    f = str(f).lower()
    if 'oil' in f and 'fuel' not in f:
        return 'Heating Oil'
    if 'gas' in f and 'lpg' not in f:
        return 'Mains Gas'
    if 'lpg' in f or 'propane' in f or 'butane' in f:
        return 'LPG'
    # check heat pump BEFORE electricity, since seai labels them like
    # 'Electric Heat Pump (Air-Water)' which would otherwise hit the electric branch first
    if 'heat pump' in f or 'heatpump' in f:
        return 'Heat Pump'
    if 'electric' in f:
        return 'Electricity'
    if 'wood' in f or 'pellet' in f or 'biomass' in f:
        return 'Wood/Pellets'
    if 'coal' in f or 'peat' in f or 'turf' in f or 'anthracite' in f or 'solid' in f:
        return 'Solid Fuel'
    return 'Other'

df['fuel_bucket'] = df['mainspaceheatingfuel'].apply(bucket_fuel)

print(sorted(df['energyrating'].dropna().unique()))
print(f'rows: {len(df):,}, columns: {len(df.columns)}')

In [ ]:
# adding the signature style

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a1a',
    'axes.grid': True,
    'grid.color': '#2a2a2a',
    'xtick.color': '#888888',
    'ytick.color': '#888888',
    'axes.labelcolor': '#cccccc',
    'axes.titlecolor': '#ffffff',
    'font.family': 'monospace',
    'axes.titlesize': 14,
    'legend.facecolor': '#1a1a1a',
    'legend.edgecolor': '#333333',
})

# rating gradient (A1 best, G worst) - same colour logic SEAI uses on the cert
RATING_COLOURS = {
    'A1': '#0a8a3f', 'A2': '#1ca84c', 'A3': '#3fc663',
    'B1': '#7fd66a', 'B2': '#b8e070', 'B3': '#e6e060',
    'C1': '#f5d24a', 'C2': '#f5b540', 'C3': '#f59340',
    'D1': '#f06f30', 'D2': '#e85528',
    'E1': '#d63a2a', 'E2': '#c8281f',
    'F':  '#a01818',
    'G':  '#6e0a0a',
}

# main accents for non-rating charts
IRELAND_GREEN = '#169b62'
WARNING_RED   = '#ff4444'
ACCENT_BLUE   = '#4a9eff'
FOSSIL_BROWN  = '#8b4513'

rating_order = ['A1','A2','A3','B1','B2','B3','C1','C2','C3','D1','D2','E1','E2','F','G']

In [ ]:
# annual heating cost per BER grade for a typical 100m² Irish home
# (rough cost - actual depends on fuel, but the GAP between grades is the point)

# blended Irish energy price - real prices vary by fuel, but most Irish homes burn
# gas or oil and ~12c/kWh is a reasonable middle ground in 2026
PRICE_PER_KWH = 0.12
TYPICAL_M2    = 100

median_ber_by_grade = df.groupby('energyrating')['berrating'].median()
median_ber_by_grade = median_ber_by_grade.reindex(rating_order).dropna()

annual_cost = median_ber_by_grade * TYPICAL_M2 * PRICE_PER_KWH

fig, ax = plt.subplots(figsize=(14, 6))
colours = [RATING_COLOURS[r] for r in median_ber_by_grade.index]
bars = ax.bar(median_ber_by_grade.index, annual_cost.values, color=colours, edgecolor='#0f0f0f', linewidth=1)

# label each bar with the euro cost
for bar, cost in zip(bars, annual_cost.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60,
            f'€{cost:,.0f}', ha='center', fontsize=9, color='#cccccc')

ax.set_title(f'Estimated Annual Heating Cost by BER Grade  ({TYPICAL_M2}m² home, €{PRICE_PER_KWH:.2f}/kWh blended)', pad=12)
ax.set_ylabel('Annual heating cost (EUR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'€{int(x):,}'))
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

a2_cost = annual_cost.get('A2', float('nan'))
g_cost  = annual_cost.get('G', float('nan'))
if not np.isnan(a2_cost) and not np.isnan(g_cost):
    print(f'A2 home heating cost:  ~€{a2_cost:,.0f}/year')
    print(f'G  home heating cost:  ~€{g_cost:,.0f}/year')
    print(f'a G-rated home costs ~€{g_cost - a2_cost:,.0f}/year more to heat than an A2')
    print(f'over a 30-year mortgage that gap is ~€{(g_cost - a2_cost) * 30:,.0f}')

In [ ]:
# rating distribution - how many homes sit at each letter grade

rating_counts = df['energyrating'].value_counts().reindex(rating_order, fill_value=0)
rating_pct = (rating_counts / rating_counts.sum()) * 100

fig, ax = plt.subplots(figsize=(14, 6))
colours = [RATING_COLOURS[r] for r in rating_order]
bars = ax.bar(rating_order, rating_counts.values, color=colours, edgecolor='#0f0f0f', linewidth=1)

# label each bar with its share of total stock
for bar, pct in zip(bars, rating_pct.values):
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                f'{pct:.1f}%', ha='center', fontsize=9, color='#cccccc')

ax.set_title('BER Distribution Across All Irish Homes', pad=12)
ax.set_ylabel('Number of dwellings')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x/1000)}k'))
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

a_share = rating_pct[['A1','A2','A3']].sum()
g_share = rating_pct[['F','G']].sum()
cd_share = rating_pct[['C1','C2','C3','D1','D2']].sum()
print(f'A-rated:  {a_share:.1f}%')
print(f'C and D:  {cd_share:.1f}%   <- the bulk of Irish housing')
print(f'F and G:  {g_share:.1f}%')

In [ ]:
# the 2008 building regs cliff - median BER by year built

yr = df[(df['year_of_construction'] >= 1900) & (df['year_of_construction'] <= 2026)].copy()

by_year = yr.groupby('year_of_construction').agg(
    median_ber=('berrating', 'median'),
    n=('berrating', 'count')
).reset_index()
by_year = by_year[by_year['n'] >= 50]

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(by_year['year_of_construction'], by_year['median_ber'], alpha=0.15, color=ACCENT_BLUE)
ax.plot(by_year['year_of_construction'], by_year['median_ber'], color=ACCENT_BLUE, linewidth=1.4, label='Median BER')

# 2008 was when Part L building regs got dramatically stricter
ax.axvline(2008, color=WARNING_RED, linewidth=1.2, linestyle='--', alpha=0.8)
ax.text(2008.3, by_year['median_ber'].max() * 0.95, '2008\nbuilding regs',
        color=WARNING_RED, fontsize=9, fontweight='bold')

ax.set_title('Median BER by Year of Construction  (lower = better)', pad=12)
ax.set_xlabel('Year built')
ax.set_ylabel('Median BER (kWh/m²/yr)')
ax.set_xlim(1900, 2026)
plt.tight_layout()
plt.show()

pre = yr[yr['year_of_construction'] < 2008]['berrating'].median()
post = yr[yr['year_of_construction'] >= 2008]['berrating'].median()
print(f'pre-2008 median:  {pre:.0f} kWh/m²/yr')
print(f'2008+ median:     {post:.0f} kWh/m²/yr')
print(f'newer homes use {(1 - post/pre)*100:.0f}% less energy per m²')

In [ ]:
# apartments vs detached - sharing walls with neighbours saves a lot of energy

def bucket_dwelling(d):
    if pd.isna(d):
        return None
    d = str(d).lower()
    if 'detached' in d and 'semi' not in d:
        return 'Detached'
    if 'semi' in d:
        return 'Semi-detached'
    if 'mid' in d and 'terr' in d:
        return 'Mid-terrace'
    if 'end' in d and 'terr' in d:
        return 'End-terrace'
    if 'apartment' in d or 'flat' in d or 'maisonette' in d:
        return 'Apartment'
    if 'bungalow' in d:
        return 'Bungalow'
    return None

df['dwelling_bucket'] = df['dwellingtypedescr'].apply(bucket_dwelling)

dwelling_order = ['Apartment', 'Mid-terrace', 'End-terrace', 'Semi-detached', 'Bungalow', 'Detached']
by_dwelling = df.dropna(subset=['dwelling_bucket']).groupby('dwelling_bucket')['berrating'].agg(['median', 'count'])
by_dwelling = by_dwelling.reindex([d for d in dwelling_order if d in by_dwelling.index])

fig, ax = plt.subplots(figsize=(12, 5))
colours = [IRELAND_GREEN if d == 'Apartment' else WARNING_RED if d == 'Detached' else ACCENT_BLUE for d in by_dwelling.index]
bars = ax.barh(by_dwelling.index, by_dwelling['median'], color=colours, alpha=0.85)

for bar, val, n in zip(bars, by_dwelling['median'], by_dwelling['count']):
    ax.text(val + 3, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}  ({n:,})', va='center', fontsize=9, color='#cccccc')

ax.set_title('Median BER by Dwelling Type  (lower = more efficient)', pad=12)
ax.set_xlabel('Median BER (kWh/m²/yr)')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

if 'Apartment' in by_dwelling.index and 'Detached' in by_dwelling.index:
    apt_med = by_dwelling.loc['Apartment', 'median']
    det_med = by_dwelling.loc['Detached', 'median']
    print(f'apartment median BER: {apt_med:.0f} kWh/m²/yr')
    print(f'detached median BER:  {det_med:.0f} kWh/m²/yr')
    print(f'apartments use {(1 - apt_med/det_med)*100:.0f}% less energy per m² than detached homes')

In [ ]:
# co2 emissions per home by decade built - the climate story directly
# (this column is reliable, heat pump column wasn't)

co2 = df[(df['year_of_construction'] >= 1900) & (df['year_of_construction'] <= 2026)].copy()
co2['co2rating'] = pd.to_numeric(co2['co2rating'], errors='coerce')
co2['decade'] = (co2['year_of_construction'] // 10 * 10).astype(int)
co2 = co2[co2['co2rating'].between(0, 200)]

co2_decade = co2.groupby('decade')['co2rating'].median()
co2_decade = co2_decade[co2_decade.index >= 1900]

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(co2_decade.index, co2_decade.values, alpha=0.2, color=IRELAND_GREEN)
ax.plot(co2_decade.index, co2_decade.values, color=IRELAND_GREEN, linewidth=1.8, marker='o', markersize=5)

ax.set_title('Median CO₂ Emissions by Decade Built  (kg CO₂/m²/yr)', pad=12)
ax.set_xlabel('Decade built')
ax.set_ylabel('kg CO₂/m²/yr')
plt.tight_layout()
plt.show()

if len(co2_decade) > 0:
    first_co2 = co2_decade.iloc[0]
    last_co2  = co2_decade.iloc[-1]
    print(f'{int(co2_decade.index[0])}s median CO2:  {first_co2:.1f} kg CO2/m²/yr')
    print(f'{int(co2_decade.index[-1])}s median CO2: {last_co2:.1f} kg CO2/m²/yr')
    print(f'modern homes emit {(1 - last_co2/first_co2)*100:.0f}% less CO2 per m²')


In [ ]:
# county leaderboard - median BER per county, best three in green and worst three in red

by_county = df.groupby('county_clean').agg(
    median_ber=('berrating', 'median'),
    n=('berrating', 'count')
).reset_index()
by_county = by_county[by_county['n'] >= 1000].sort_values('median_ber')

best_three  = by_county['county_clean'].head(3).tolist()
worst_three = by_county['county_clean'].tail(3).tolist()
colours = [IRELAND_GREEN if c in best_three else WARNING_RED if c in worst_three else ACCENT_BLUE
           for c in by_county['county_clean']]

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(by_county['county_clean'], by_county['median_ber'], color=colours, alpha=0.85)

for bar, val in zip(bars, by_county['median_ber']):
    ax.text(val + 3, bar.get_y() + bar.get_height()/2, f'{val:.0f}',
            va='center', fontsize=8, color='#cccccc')

ax.set_title('Median BER by County  (lower = more efficient)', pad=12)
ax.set_xlabel('Median BER (kWh/m²/yr)')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

print('most efficient counties:')
for _, row in by_county.head(3).iterrows():
    print(f'  {row["county_clean"]:<20} {row["median_ber"]:.0f}  ({row["n"]:,} ratings)')
print('\nleast efficient counties:')
for _, row in by_county.tail(3).iterrows():
    print(f'  {row["county_clean"]:<20} {row["median_ber"]:.0f}  ({row["n"]:,} ratings)')

In [ ]:
# fuel type by decade built - the energy transition story in one chart

yr2 = df[(df['year_of_construction'] >= 1900) & (df['year_of_construction'] <= 2026)].copy()
yr2['decade'] = (yr2['year_of_construction'] // 10 * 10).astype(int)
yr2 = yr2[yr2['decade'] >= 1900]

fuel_decade = yr2.groupby(['decade', 'fuel_bucket']).size().unstack(fill_value=0)
fuel_decade_pct = fuel_decade.div(fuel_decade.sum(axis=1), axis=0) * 100

fuel_order = ['Solid Fuel', 'Heating Oil', 'LPG', 'Mains Gas', 'Electricity', 'Heat Pump', 'Wood/Pellets', 'Other', 'Unknown']
fuel_order = [f for f in fuel_order if f in fuel_decade_pct.columns]
fuel_decade_pct = fuel_decade_pct[fuel_order]

fuel_palette = {
    'Solid Fuel':   '#3a3a3a',
    'Heating Oil':  FOSSIL_BROWN,
    'LPG':          '#a07c2a',
    'Mains Gas':    ACCENT_BLUE,
    'Electricity':  '#f5d24a',
    'Heat Pump':    IRELAND_GREEN,
    'Wood/Pellets': '#7a5230',
    'Other':        '#888888',
    'Unknown':      '#444444',
}

fig, ax = plt.subplots(figsize=(14, 7))
fuel_decade_pct.plot(kind='bar', stacked=True, ax=ax,
                     color=[fuel_palette[f] for f in fuel_order],
                     width=0.85, edgecolor='#0f0f0f', linewidth=0.5)
ax.set_title('Main Heating Fuel by Decade Built', pad=12)
ax.set_xlabel('Decade built')
ax.set_ylabel('% of dwellings')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=True)
ax.set_xticklabels([f'{int(d)}s' for d in fuel_decade_pct.index], rotation=45)
plt.tight_layout()
plt.show()

fossil = ['Heating Oil', 'LPG', 'Mains Gas', 'Solid Fuel']
fossil_overall = df['fuel_bucket'].isin(fossil).mean() * 100
print(f'overall % of Irish homes on fossil fuel heat: {fossil_overall:.1f}%')
if 'Heat Pump' in fuel_decade_pct.columns and 2020 in fuel_decade_pct.index:
    hp_recent = fuel_decade_pct.loc[2020:, 'Heat Pump'].mean()
    print(f'heat pumps in homes built 2020s: {hp_recent:.1f}%')

In [ ]:
# wall insulation across a century - measured by wall U-value
# (lower u-value = more insulating, so the line going down is good)

uv = df[(df['year_of_construction'] >= 1900) & (df['year_of_construction'] <= 2026)].copy()
uv['decade'] = (uv['year_of_construction'] // 10 * 10).astype(int)
uv = uv[uv['uvaluewall'].between(0.05, 5)]

uv_decade = uv.groupby('decade')['uvaluewall'].median()
uv_decade = uv_decade[uv_decade.index >= 1900]

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(uv_decade.index, uv_decade.values, alpha=0.2, color=ACCENT_BLUE)
ax.plot(uv_decade.index, uv_decade.values, color=ACCENT_BLUE, linewidth=1.8, marker='o', markersize=5)

ax.set_title('Median Wall U-Value by Decade Built  (lower = more insulating)', pad=12)
ax.set_xlabel('Decade built')
ax.set_ylabel('Wall U-value (W/m²K)')
plt.tight_layout()
plt.show()

if len(uv_decade) > 0:
    first_u = uv_decade.iloc[0]
    last_u  = uv_decade.iloc[-1]
    print(f'{int(uv_decade.index[0])}s median wall U-value:  {first_u:.2f} W/m²K')
    print(f'{int(uv_decade.index[-1])}s median wall U-value: {last_u:.2f} W/m²K')
    print(f'modern walls lose {(1 - last_u/first_u)*100:.0f}% less heat per m² of wall area')

In [ ]:
# where the worst-rated (F+G) homes actually live

worst = df[df['energyrating'].isin(['F', 'G'])].copy()
worst = worst[(worst['year_of_construction'] >= 1900) & (worst['year_of_construction'] <= 2026)]
worst['decade'] = (worst['year_of_construction'] // 10 * 10).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# left panel - F+G homes by decade
by_decade = worst.groupby('decade').size()
by_decade = by_decade[by_decade.index >= 1900]
axes[0].bar(by_decade.index, by_decade.values, color=WARNING_RED, alpha=0.85, width=8)
axes[0].set_title('F+G Rated Homes by Decade Built', pad=12)
axes[0].set_xlabel('Decade built')
axes[0].set_ylabel('Number of dwellings')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x/1000)}k'))

# right panel - top 10 counties for F+G stock
by_county_worst = worst.groupby('county_clean').size().sort_values(ascending=True).tail(10)
axes[1].barh(by_county_worst.index, by_county_worst.values, color=WARNING_RED, alpha=0.85)
axes[1].set_title('Top 10 Counties by F+G Rated Homes', pad=12)
axes[1].set_xlabel('Number of dwellings')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x/1000)}k'))
axes[1].grid(axis='y', visible=False)

plt.tight_layout()
plt.show()

total_worst = len(worst)
worst_pct = total_worst / len(df) * 100
print(f'F or G rated homes in Ireland: {total_worst:,} ({worst_pct:.1f}% of total)')
if len(by_decade) > 0:
    peak_decade = by_decade.idxmax()
    print(f'worst decade for F+G stock: {int(peak_decade)}s ({by_decade.max():,} dwellings)')